<a href="https://colab.research.google.com/github/shaan-byte/python_ds_colab/blob/main/API_Fundamentals_and_REST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Session 4.1 — API Fundamentals & REST

**Module 1: Programming & Data Foundations**  
**Week 4 | Session 4.1**

---

## What we will cover today

1. What is an API and why does it exist?
2. The Request-Response cycle
3. HTTP methods — GET and POST
4. URL parameters
5. REST architecture — what makes an API "RESTful"?

---

> **The goal of this session:** By the end, you should be able to write Python code that fetches data from a real API on the internet, understands what came back, and knows how to ask the API for something more specific.

---
## Setup — Run this first

We need the `requests` library. It is pre-installed in Google Colab, so this should work immediately.

In [ ]:
import requests
import json

print("requests version:", requests.__version__)
print("Ready to go!")

requests version: 2.32.4
Ready to go!


---
## Part 1 — What is an API?

### The restaurant analogy

Imagine you walk into a restaurant.

- You (the customer) want food.
- The kitchen has the food.
- But you cannot walk into the kitchen yourself and start cooking.

So there is a **waiter**. You tell the waiter what you want. The waiter goes to the kitchen, gets it, and brings it back to you.

An **API (Application Programming Interface)** is the waiter.

| Restaurant | Software World |
|---|---|
| You (customer) | Your Python program |
| The kitchen | A server with data or functionality |
| The waiter | The API |
| The menu | API documentation (what you are allowed to ask for) |
| Your order | An API request |
| The dish that arrives | The API response |

---

### Why do APIs exist?

Consider these real scenarios:

**Scenario A:** You are building a weather app. You do not have weather stations across the world. But the Indian Meteorological Department does. They expose an API so that your app can ask: "What is the weather in Mumbai right now?" and get an answer.

**Scenario B:** A bank wants to let third-party apps check your account balance. They cannot give those apps direct access to their database. So they expose a tightly controlled API that returns only what is needed.

**Scenario C:** As a data scientist, you want information about all the countries in the world — population, languages, currencies. You could scrape Wikipedia. Or you could call an API that gives you clean, structured JSON in one line of code.

APIs let different software systems **talk to each other** in a controlled, standardised way.

---

---
## Part 2 — The Request-Response Cycle

Every interaction with an API follows the same pattern. Always.

```
Your Program                     Server (API)
    |                                 |
    |  --- REQUEST ----------------> |
    |                                 |  (server processes your request)
    |  <-- RESPONSE ----------------  |
    |                                 |
```

### What is in a Request?

A request has four main parts:

| Part | What it is | Example |
|---|---|---|
| **URL** | The address of the resource you want | `https://restcountries.com/v3.1/name/india` |
| **Method** | What action you want to perform | `GET` (fetch data) |
| **Headers** | Extra metadata about the request | `Content-Type: application/json` |
| **Body** | Data you are sending (for POST) | `{"username": "ananya"}` |

### What is in a Response?

| Part | What it is | Example |
|---|---|---|
| **Status code** | A number that tells you if it worked | `200` (success), `404` (not found) |
| **Headers** | Metadata about the response | `Content-Type: application/json` |
| **Body** | The actual data the server sent back | JSON with country information |

We will see all of these in action in a moment.

---
## Part 3 — Your First API Call

### The API we will use today

We will use the **REST Countries API** — a free, public API that gives information about every country in the world. No account, no API key needed.

Base URL: `https://restcountries.com/v3.1`

Let's ask it a simple question: **"Tell me about India."

In [ ]:
# Make your first API call
# We are asking the REST Countries API for information about India
from google.colab import userdata

url = "https://api.restcountries.com/countries/v5?limit=1"

response = requests.get(
    url,
    headers={
        'Authorization': userdata.get('con_key')
    })

# Let's see what came back
print("Status code:", response.status_code)
print("Content type:", response.headers.get("Content-Type"))
print("Response size:", response.text, "characters")

Status code: 200
Content type: application/json
Response size: {"data":{"objects":[{"names":{"alternates":["Apsny"],"common":"Abkhazia","native":{"abk":{"common":"Аҧсны","official":"Аҧсны Аҳәынҭқарра"},"rus":{"common":"Абхазия","official":"Республика Абхазия"}},"official":"Republic of Abkhazia","translations":{"ara":{"common":"أبخازيا","official":"جمهورية أبخازيا"},"deu":{"common":"Abchasien","official":"Republik Abchasien"},"fra":{"common":"Abkhazie","official":"République d'Abkhazie"},"hun":{"common":"Abházia","official":"Abház Köztársaság"},"ita":{"common":"Abcasia","official":"Repubblica di Abcasia"},"jpn":{"common":"アブハジア","official":"アブハジア共和国"},"kor":{"common":"압하지야","official":"압하지야 공화국"},"nld":{"common":"Abchazië","official":"Republiek Abchazië"},"pol":{"common":"Abchazja","official":"Republika Abchazji"},"por":{"common":"Abecásia","official":"República da Abecásia"},"rus":{"common":"Абхазия","official":"Республика Абхазия"},"spa":{"common":"Abjasia","official":"República de Ab

In [ ]:
data = response.json()
data[0]['capital']

['New Delhi']

### Understanding status codes

The status code is the server's way of telling you whether your request worked, and if not, why.

They are grouped into ranges — the first digit tells you the category:

| Range | Category | Meaning |
|---|---|---|
| 2xx | Success | Request worked |
| 3xx | Redirection | The resource has moved |
| 4xx | Client error | You made a mistake in the request |
| 5xx | Server error | The server had a problem |

The most common ones you will see:

| Code | Meaning | When you see it |
|---|---|---|
| `200` | OK | Everything worked |
| `201` | Created | You POSTed something and it was created |
| `400` | Bad Request | You sent something malformed |
| `401` | Unauthorized | You need to authenticate |
| `403` | Forbidden | You authenticated but do not have permission |
| `404` | Not Found | The URL or resource does not exist |
| `429` | Too Many Requests | You are being rate-limited |
| `500` | Internal Server Error | Something broke on the server's side |

In [ ]:
# The response body is JSON. Let's parse it and look at the structure.
# response.json() converts the JSON text into a Python list/dict automatically.

data = response.json()

print("Type of data:", type(data))
print("Number of results:", len(data))
print()
print("Top-level keys in the first result:")
for key in list(data[0].keys()):
    print(f"  {key}")

In [ ]:
# There is a lot of data. Let's extract the fields we actually care about.

country = data[0]

name       = country["name"]["common"]
capital    = country["capital"][0] if country.get("capital") else "N/A"
population = country["population"]
region     = country["region"]
area       = country["area"]
languages  = list(country["languages"].values())
currencies = [v["name"] for v in country["currencies"].values()]

print(f"Country:    {name}")
print(f"Capital:    {capital}")
print(f"Population: {population:,}")
print(f"Region:     {region}")
print(f"Area:       {area:,} sq km")
print(f"Languages:  {', '.join(languages)}")
print(f"Currencies: {', '.join(currencies)}")

### Live demo — try other countries

The cell below has a variable `country_name` you can change to any country. Try it during class.

In [ ]:
# Change this to any country name and re-run the cell
country_name = "india"

url = f"https://restcountries.com/v3.1/name/{country_name}"
response = requests.get(url)

if response.status_code == 200:
    data = response.json()
    print(data)
    for country in data:
      # print(f"Name:       {country['name']['common']}")
      print(f"Capital:    {country.get('capital', ['N/A'])[0]}")
      print(f"Population: {country['population']:,}")
      print(f"Region:     {country['region']}")
      print(f"Languages:  {', '.join(country.get('languages', {}).values())}")
    # country = response.json()[0]
elif response.status_code == 404:
    print(f"Country '{country_name}' not found. Check the spelling and try again.")
else:
    print(f"Something went wrong. Status code: {response.status_code}")

{'success': False, 'data': None, 'errors': [{'message': 'This API version has been deprecated. Please visit the https://restcountries.com/docs/legacy-api-deprecation to migrate to our new version (v5).'}]}


AttributeError: 'str' object has no attribute 'get'

In [ ]:
d = {
    "country": "India"
}

In [ ]:
d['eountry']

KeyError: 'eountry'

In [ ]:
d.get('country')

'India'

---
### Exercise 1 — Making Your First API Call

**Task:** Complete the function below. It should call the REST Countries API for a given country name and return a dictionary with three fields: `name`, `capital`, and `population`.

If the country is not found (status code 404), return `None`.

In [ ]:
def get_country_info(country_name):
    """
    Fetches basic info for a country from the REST Countries API.
    Returns a dict with 'name', 'capital', 'population'.
    Returns None if the country is not found.
    """
    url = f"https://restcountries.com/v3.1/name/{country_name}"   # <-- which variable goes here?

    response = requests.get(url)                        # <-- which method? what URL?

    if response.status_code ==200:                       # <-- what code means success?
        country = response.json()[0]
        return {
            "name":       country["name"]["common"],
            "capital":    country.get("capital", ["N/A"])[0],
            "population": country["population"]                    # <-- which key holds population?
        }
    elif response.status_code == 404:
        return None                                        # <-- what to return if not found?
    else:
        return None


# Test it
result = get_country_info("singapore")
print("Singapore result:", result)

result2 = get_country_info("zzzfakecountry")
print("Fake country result:", result2)

print()
print(f"Correct (Singapore found):     {result is not None and result['name'] == 'Singapore'}")
print(f"Correct (fake country = None): {result2 is None}")

<details>
<summary>Show solution</summary>

```python
def get_country_info(country_name):
    url = f"https://restcountries.com/v3.1/name/{country_name}"
    response = requests.get(url)
    if response.status_code == 200:
        country = response.json()[0]
        return {
            "name":       country["name"]["common"],
            "capital":    country.get("capital", ["N/A"])[0],
            "population": country["population"]
        }
    elif response.status_code == 404:
        return None
    else:
        return None
```

</details>

---
## Part 4 — URL Parameters

### What is a URL?

Before we talk about parameters, let's understand the structure of a URL. Every URL has parts:

```
https://restcountries.com/v3.1/name/india?fields=name,capital,population
  |          |              |       |    |        |
 scheme    host           path  resource  ?   query string
```

| Part | Example | Purpose |
|---|---|---|
| Scheme | `https://` | Protocol to use |
| Host | `restcountries.com` | Which server to contact |
| Path | `/v3.1/name/` | Which part of the server |
| Resource | `india` | Which specific resource |
| Query string | `?fields=name,capital` | Extra instructions |

---

### Two ways to pass parameters

**1. Path parameters** — baked directly into the URL path. Used to identify a specific resource.

```
https://restcountries.com/v3.1/name/india
                                    ^^^^^
                              path parameter: which country?
```

**2. Query parameters** — appended after a `?`, as `key=value` pairs separated by `&`. Used to filter, sort, or customise the response.

```
https://restcountries.com/v3.1/name/india?fields=name,capital&fullText=true
                                          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
                                          query parameters
```

The `?` marks the start. Each parameter is `key=value`. Multiple parameters are separated by `&`.

---

### How `requests` handles query parameters

You could build the URL string yourself, but `requests` gives you a cleaner way using the `params` argument. It handles the encoding for you (e.g. spaces become `%20`).

In [ ]:
# The REST Countries API supports a 'fields' query parameter
# that lets you ask for only specific fields — useful for reducing response size

# Without params — the full response
url = "https://restcountries.com/v3.1/name/brazil"
full_response = requests.get(url)
full_data = full_response.json()[0]
print(f"Full response has {len(full_data)} fields")

print()

# With params — only ask for the fields we need
params = {
    "fields": "name,capital,population,region,languages"
}

filtered_response = requests.get(url, params=params)
filtered_data = filtered_response.json()

# NOTE: when using 'fields', the API may return a list or a single object
if isinstance(filtered_data, list):
    filtered_data = filtered_data[0]

print(f"Filtered response has {len(filtered_data)} fields")
print()
print("Actual URL that was called:")
print(" ", filtered_response.url)
print()
print("Filtered data:")
for key, value in filtered_data.items():
    print(f"  {key}: {value}")

Full response has 35 fields

Filtered response has 5 fields

Actual URL that was called:
  https://restcountries.com/v3.1/name/brazil?fields=name%2Ccapital%2Cpopulation%2Cregion%2Clanguages

Filtered data:
  name: {'common': 'Brazil', 'official': 'Federative Republic of Brazil', 'nativeName': {'por': {'official': 'República Federativa do Brasil', 'common': 'Brasil'}}}
  languages: {'por': 'Portuguese'}
  capital: ['Brasília']
  region: Americas
  population: 213421037


In [ ]:
# The API also lets us filter by region — fetch all countries in Asia
# and use query params to get only the fields we want

url = "https://restcountries.com/v3.1/region/asia"
params = {"fields": "name,population,capital"}

response = requests.get(url, params=params)
asian_countries = response.json()

print(f"Number of countries in Asia: {len(asian_countries)}")
print()

# Sort by population descending and show the top 8
sorted_countries = sorted(
    asian_countries,
    key=lambda c: c.get("population", 0),
    reverse=True
)

print("Top 8 most populous countries in Asia:")
print(f"  {'Rank':<5} {'Country':<25} {'Population':>15} {'Capital':<20}")
print("  " + "-" * 68)
for i, c in enumerate(sorted_countries[:8], start=1):
    name    = c["name"]["common"]
    pop     = c.get("population", 0)
    capital = c.get("capital", ["N/A"])[0]
    print(f"  {i:<5} {name:<25} {pop:>15,} {capital:<20}")

---
### Exercise 2 — Using URL Parameters

**Task:** Complete the function below. It should fetch all countries in a given **region** (e.g. `"europe"`, `"africa"`) and return a list of their names, sorted alphabetically.

Use the `params` argument to request only the `name` field — we do not need anything else.

In [ ]:
def get_countries_in_region(region):
    """
    Returns an alphabetically sorted list of country names in the given region.
    """
    url = f"https://restcountries.com/v3.1/region/{region}"  # <-- which variable?

    params = {
        "fields":"name"   # <-- we only need the 'name' field
    }

    response = requests.get(url, params=params)              # <-- fill both arguments

    if response.status_code != 200:
        return []

    countries = response.json()

    # Extract the common name from each country and sort
    names = [c["name"]["common"] for c in countries]
    return sorted(names)                                    # <-- what do we sort?


# Test it
african_countries = get_countries_in_region("africa")
print(f"Countries in Africa: {len(african_countries)}")
print(f"First 5 alphabetically: {african_countries[:5]}")
print(f"Last  5 alphabetically: {african_countries[-5:]}")
print()
print(f"Correct (more than 40 countries found): {len(african_countries) > 40}")
print(f"Correct (sorted alphabetically):        {'Algeria' in african_countries[:5]}")

<details>
<summary>Show solution</summary>

```python
def get_countries_in_region(region):
    url = f"https://restcountries.com/v3.1/region/{region}"
    params = {"fields": "name"}
    response = requests.get(url, params=params)
    if response.status_code != 200:
        return []
    countries = response.json()
    names = [c["name"]["common"] for c in countries]
    return sorted(names)
```

</details>

---
## Part 5 — HTTP Methods: GET vs POST

### The key idea

An HTTP method tells the server **what you want to do**, not just **where** you want to do it.

Think of it this way — the URL is like a **noun** (what thing), and the method is the **verb** (what action).

```
GET  /employees/101   →  "Give me employee 101"
POST /employees       →  "Create a new employee"
```

There are several HTTP methods, but the two most important ones for now are:

---

### GET — Fetching data

- Used to **retrieve** information from the server
- Data you send goes in the URL (as query parameters)
- Should never change anything on the server
- Can be cached, bookmarked, and repeated safely
- You have already been using GET in every example so far

**Real-world analogies:**
- Searching for a product on Amazon
- Loading your Twitter feed
- Checking your bank balance (read-only)

---

### POST — Sending data to create something

- Used to **send** data to the server to create something new
- Data goes in the **request body** (not the URL)
- Can (and usually does) change something on the server
- Not safe to repeat blindly — clicking submit twice might create two orders

**Real-world analogies:**
- Placing an order on Swiggy
- Submitting a job application form
- Sending a message on WhatsApp
- Creating an account on a website

---

### The key difference: where does the data go?

| | GET | POST |
|---|---|---|
| Data location | URL query string | Request body |
| Visible in browser | Yes (in the URL bar) | No |
| Length limit | Yes (URLs have limits) | No |
| Use case | Fetching | Creating / Submitting |
| Safe to repeat? | Yes | No |

---

### GET vs POST

For each action below, decide: is it a GET or a POST?

1. Loading the Flipkart homepage
2. Submitting a login form with username and password
3. Searching for "Python books" on Google
4. Placing an order on Amazon
5. Checking the score of yesterday's cricket match
6. Posting a comment on YouTube

---
### Demonstrating POST with HTTPBin

We cannot demonstrate POST with the REST Countries API — it is a read-only API. We need a server that accepts data we send.

**HTTPBin** (`httpbin.org`) is a tool built specifically for this purpose. It is a free, public service that:
- Accepts any HTTP request
- **Mirrors back exactly what you sent** in the response

This makes it perfect for learning — you can see exactly what your request looked like from the server's perspective.

In [ ]:
# First, a GET request to HTTPBin
# /get endpoint echoes back what it received

params = {
    "search": "python course",
    "page": "1",
    "limit": "10"
}

response = requests.get("https://httpbin.org/get", params=params)
print("Status code:", response.status_code)
data = response.json()

print("Status code:", response.status_code)
print()
print("URL that was called:")
print(" ", response.url)
print()
print("Query parameters the server received:")
for key, value in data["args"].items():
    print(f"  {key} = {value}")
print()
print("Some headers the server received:")
useful_headers = ["User-Agent", "Accept", "Host"]
for h in useful_headers:
    if h in data["headers"]:
        print(f"  {h}: {data['headers'][h]}")

Status code: 200
Status code: 200

URL that was called:
  https://httpbin.org/get?search=python+course&page=1&limit=10

Query parameters the server received:
  limit = 10
  page = 1
  search = python course

Some headers the server received:
  User-Agent: python-requests/2.32.4
  Accept: */*
  Host: httpbin.org


In [ ]:
# Now a POST request — sending data in the request body
# Imagine this is a form submission: "create a new employee record"

# This is the data we are sending
new_employee = {
    "name":       "Ananya Sharma",
    "department": "Engineering",
    "salary":     72000,
    "join_date":  "2024-06-01"
}

# We use requests.post() and pass the data as 'json='
# This tells requests to:
#   1. Serialize our dictionary as a JSON string
#   2. Set the Content-Type header to 'application/json'
response = requests.post("https://httpbin.org/post", json=new_employee)

echo = response.json()

print("Status code:", response.status_code)
print()
print("Data the server received in the request body:")
received_data = json.loads(echo["data"])   # the body is a JSON string inside the JSON response
for key, value in received_data.items():
    print(f"  {key}: {value}")
print()
print("Content-Type header sent:", echo["headers"].get("Content-Type"))

Status code: 200

Data the server received in the request body:
  name: Ananya Sharma
  department: Engineering
  salary: 72000
  join_date: 2024-06-01

Content-Type header sent: application/json


### Side by side: what changed between GET and POST?

Run both cells above and compare:

- In the GET request, the parameters appear in `args` (from the URL)
- In the POST request, the data appears in `data` (from the request body)
- The GET URL had `?search=python+course&page=1&limit=10` visible in the URL
- The POST URL was just `https://httpbin.org/post` — no data visible in the URL

---
### Exercise 3 — GET vs POST

**Task:** Two functions below are incomplete. The first should make a GET request. The second should make a POST request. Complete the blanks.

Both functions use HTTPBin so you can verify from the response that the data arrived correctly.

In [ ]:
# Part A: Complete the GET request
def search_products(category, max_price):
    """
    Simulates a product search by sending query params to HTTPBin.
    Returns the args dict from the response (what the server received).
    """
    params = {
        "category":  ___,       # <-- which variable?
        "max_price": ___        # <-- which variable?
    }
    response = requests.___("https://httpbin.org/get", params=___)   # <-- GET or POST? params= or json=?
    return response.json()["args"]


# Part B: Complete the POST request
def submit_feedback(user_name, rating, comment):
    """
    Simulates submitting a feedback form by POSTing JSON to HTTPBin.
    Returns the data dict from the response (what the server received in the body).
    """
    payload = {
        "user_name": user_name,
        "rating":    rating,
        "comment":   comment
    }
    response = requests.___("https://httpbin.org/post", ___=payload)  # <-- GET or POST? params= or json=?
    return json.loads(response.json()["data"])


# Test Part A
search_result = search_products(category="electronics", max_price=5000)
print("Search params received by server:", search_result)
print(f"Correct: {search_result.get('category') == 'electronics' and search_result.get('max_price') == '5000'}")

print()

# Test Part B
feedback_result = submit_feedback(user_name="Rohan", rating=5, comment="Great session!")
print("Feedback body received by server:", feedback_result)
print(f"Correct: {feedback_result.get('user_name') == 'Rohan' and feedback_result.get('rating') == 5}")

<details>
<summary>Show solution</summary>

```python
def search_products(category, max_price):
    params = {"category": category, "max_price": max_price}
    response = requests.get("https://httpbin.org/get", params=params)
    return response.json()["args"]

def submit_feedback(user_name, rating, comment):
    payload = {"user_name": user_name, "rating": rating, "comment": comment}
    response = requests.post("https://httpbin.org/post", json=payload)
    return json.loads(response.json()["data"])
```

</details>

---
## Part 6 — REST Architecture

### What does REST mean?

**REST** stands for **Representational State Transfer**. It is not a technology or a protocol — it is a set of **design principles** for building APIs.

An API that follows these principles is called a **RESTful API**.

Most APIs you will encounter as a data scientist — weather APIs, finance APIs, social media APIs — are RESTful. Understanding these principles helps you quickly understand how any new API is organised, even before reading its documentation.

---

### The four core principles of REST

#### Principle 1: Everything is a Resource

In REST, every piece of data the API can give you is called a **resource**. A resource is a noun — a thing.

- A country is a resource
- An employee is a resource
- An order is a resource
- A product is a resource

Each resource has a **URL** that identifies it uniquely:

```
/countries/india        →  the India resource
/employees/101          →  employee number 101
/orders/ORD-9832        →  order ORD-9832
/products/SKU-4421      →  product SKU-4421
```

Notice: the URL is a **noun**. The HTTP method is the **verb**.

```
GET  /employees/101   →  "Fetch employee 101"
POST /employees       →  "Create a new employee"
```

---

#### Principle 2: Statelessness

This is one of the most important — and often confusing — principles.

**Stateless** means: the server **does not remember** anything about you between requests. Every request must contain all the information the server needs to handle it.

**Analogy — Stateful (like a regular conversation):**
> You: "I'm Ananya."
> Waiter: "Hi Ananya!"
> You: "I'll have the dosa."
> Waiter: "One dosa for Ananya, coming up!"

The waiter remembered your name. This is stateful.

**Analogy — Stateless (like REST):**
> You: "I'll have the dosa."
> Waiter: "Who are you and where are you sitting?"
> You: "I'm Ananya, table 4, here is my order token."
> Waiter: "One dosa for Ananya at table 4, coming up!"

Every request must be **self-contained**. This is what authentication tokens, API keys, and user IDs in requests are for — they identify who is asking, because the server does not remember.

**Why does this matter?** Stateless APIs are much easier to scale. If the server does not store any session state, you can add more servers without worrying about keeping sessions in sync.

---

#### Principle 3: Uniform Interface — Use HTTP methods correctly

REST standardises what each HTTP method means. When an API is RESTful, you can predict what each method does on any resource:

| Method | Action | Example |
|---|---|---|
| GET | Fetch a resource | `GET /employees/101` |
| POST | Create a new resource | `POST /employees` (with body) |
| PUT | Replace a resource entirely | `PUT /employees/101` (with full updated record) |
| PATCH | Update part of a resource | `PATCH /employees/101` (with just the changed fields) |
| DELETE | Remove a resource | `DELETE /employees/101` |

---

#### Principle 4: Responses are representations

When you fetch a resource, the server does not send you the resource itself (it cannot — you cannot physically receive a country). It sends you a **representation** of the resource.

The most common representation format is **JSON**. That is what REST Countries API returns — a JSON representation of country data.

Other formats include XML, HTML, and plain text, but JSON is the dominant format for modern APIs.

---

### What makes an API non-RESTful?

Common violations you might see in the wild:

- URLs that use verbs: `/getEmployee?id=101` instead of `GET /employees/101`
- Using GET for actions that change data: `/deleteOrder?id=5`
- Storing session state on the server between requests

These are not always wrong in practice — but they deviate from REST conventions and make the API harder to understand and use consistently.

In [ ]:
# Let's see REST principles in action using the Countries API
# Notice how the URL structure follows REST conventions

print("REST Countries API — URL structure examples")
print("=" * 55)

examples = [
    ("GET", "https://restcountries.com/v3.1/all?fields=name",          "Fetch all country resources"),
    ("GET", "https://restcountries.com/v3.1/name/india",               "Fetch the 'india' resource by name"),
    ("GET", "https://restcountries.com/v3.1/alpha/JP",                 "Fetch the resource with code 'JP' (Japan)"),
    ("GET", "https://restcountries.com/v3.1/region/asia?fields=name",  "Fetch all resources in the 'asia' region"),
    ("GET", "https://restcountries.com/v3.1/lang/hindi",               "Fetch all resources where language is Hindi"),
]

for method, url, description in examples:
    print(f"  {method:<6} {url}")
    print(f"         -> {description}")
    print()

print()
print("Notice: URLs are all nouns. The HTTP method (GET) is the verb.")
print("Notice: Each URL uniquely identifies a resource or a collection.")

In [ ]:
# Demonstrating statelessness
# Each request is completely independent — no memory between calls

print("Demonstrating statelessness: three independent requests")
print("=" * 55)

countries_to_fetch = ["india", "germany", "brazil"]

for country_name in countries_to_fetch:
    # Each of these is a self-contained request
    # The server does not know we called it before, or what we asked for before
    # Every request stands completely on its own
    url = f"https://restcountries.com/v3.1/name/{country_name}"
    params = {"fields": "name,population"}
    r = requests.get(url, params=params)
    d = r.json()
    if isinstance(d, list):
        d = d[0]
    name = d["name"]["common"]
    pop  = d.get("population", "N/A")
    print(f"  Request for '{country_name}' -> {name}, population: {pop:,}")

print()
print("Each request contained all the information needed: the URL.")
print("The server did not remember request 1 when handling request 2.")

---
### Exercise 4 — REST Design: Spot the Violation

**Task:** Look at each API design below. For each one, decide whether it follows REST principles or violates them. If it violates a principle, identify which one and suggest a better design.

Fill in the `verdict` and `suggestion` variables.

In [ ]:
# For each design below, discuss as a class.
# Then update the verdict: "good" or the principle it violates.

designs = [
    {
        "description": "GET /getProductDetails?productId=55",
        "verdict":     None,   # <-- 'good' or name the principle violated
        "suggestion":  None    # <-- what would the REST-correct version be?
    },
    {
        "description": "GET /products/55",
        "verdict":     None,
        "suggestion":  None
    },
    {
        "description": "GET /deleteOrder?id=99  (this endpoint deletes the order)",
        "verdict":     None,
        "suggestion":  None
    },
    {
        "description": "POST /orders  with body {customer_id, items, address}",
        "verdict":     None,
        "suggestion":  None
    },
    {
        "description": "POST /createNewUserAccount  with body {name, email}",
        "verdict":     None,
        "suggestion":  None
    },
]

for d in designs:
    print(f"Design:     {d['description']}")
    print(f"Verdict:    {d['verdict']}")
    print(f"Suggestion: {d['suggestion']}")
    print()

<details>
<summary>Show answers for class discussion</summary>

1. `GET /getProductDetails?productId=55`  
   **Violates:** Uniform interface — URL uses a verb (`getProductDetails`), not a noun  
   **Better:** `GET /products/55`

2. `GET /products/55`  
   **Good.** Noun URL, correct method for fetching.

3. `GET /deleteOrder?id=99`  
   **Violates two principles:**  
   - Uniform interface: GET should never cause side effects (deletion)  
   - Uniform interface: URL uses a verb  
   **Better:** `DELETE /orders/99`

4. `POST /orders` with body `{customer_id, items, address}`  
   **Good.** Noun URL, POST used to create a new resource, data in body.

5. `POST /createNewUserAccount` with body `{name, email}`  
   **Violates:** Uniform interface — URL uses a verb  
   **Better:** `POST /users` with body `{name, email}`

</details>

---
## Part 7 — Putting It All Together

Let's build something useful using everything from today. We will write a small country comparison tool that:
1. Accepts a list of country names
2. Fetches data for each one via GET with query parameters
3. Handles errors gracefully (what if a country name is wrong?)
4. Returns a sorted comparison table

In [ ]:
def compare_countries(country_names):
    """
    Fetches and compares a list of countries.
    Returns a list of result dicts, sorted by population descending.
    """
    results = []

    for name in country_names:
        url    = f"https://restcountries.com/v3.1/name/{name}"
        params = {"fields": "name,population,area,region,languages,currencies"}

        response = requests.get(url, params=params)

        if response.status_code == 200:
            c = response.json()
            if isinstance(c, list):
                c = c[0]
            languages  = list(c.get("languages", {}).values())
            currencies = [v["name"] for v in c.get("currencies", {}).values()]
            results.append({
                "name":       c["name"]["common"],
                "region":     c.get("region", "N/A"),
                "population": c.get("population", 0),
                "area_km2":   c.get("area", 0),
                "languages":  ", ".join(languages[:2]),
                "currency":   currencies[0] if currencies else "N/A"
            })
        elif response.status_code == 404:
            print(f"  Warning: '{name}' not found — skipping")
        else:
            print(f"  Warning: '{name}' returned status {response.status_code} — skipping")

    return sorted(results, key=lambda x: x["population"], reverse=True)


# Compare a set of countries
countries = ["india", "china", "united states", "indonesia", "brazil", "pakistan", "unknownland"]

comparison = compare_countries(countries)

print()
print(f"{'Country':<20} {'Region':<15} {'Population':>15} {'Area (km2)':>12} {'Language(s)':<25} {'Currency':<25}")
print("-" * 115)
for c in comparison:
    print(f"{c['name']:<20} {c['region']:<15} {c['population']:>15,} {c['area_km2']:>12,.0f} {c['languages']:<25} {c['currency']:<25}")

---
## Final Exercise — End-to-End Challenge

This exercise brings together everything from today: GET requests, URL parameters, POST requests, and reading the response correctly.

**Scenario:** You are building a simple data collection tool. It needs to:

1. Fetch the top 3 most populous countries in **South America** from the REST Countries API
2. Format them as a list of dictionaries
3. POST that list to HTTPBin as if you were submitting a report
4. Confirm that HTTPBin received the data correctly by reading the response

Complete the blanks in the cells below.

In [ ]:
# Step 1: Fetch countries in South America with the fields we need

url    = "https://restcountries.com/v3.1/region/___"        # <-- which region?
params = {"fields": "name,population,capital"}

response = requests.___(url, params=params)                  # <-- which method?

if response.status_code == 200:
    all_countries = response.json()
    print(f"Fetched {len(all_countries)} countries from South America")
else:
    print(f"Error: {response.status_code}")
    all_countries = []

In [ ]:
# Step 2: Sort by population and take the top 3

sorted_countries = sorted(
    all_countries,
    key=lambda c: c.get(___, 0),    # <-- sort by which key?
    reverse=True
)

top3 = []
for c in sorted_countries[:3]:      # top 3
    top3.append({
        "name":       c["name"]["common"],
        "capital":    c.get("capital", ["N/A"])[0],
        "population": c["population"]
    })

print("Top 3 most populous countries in South America:")
for c in top3:
    print(f"  {c['name']:<20} pop: {c['population']:>12,}  capital: {c['capital']}")

In [ ]:
# Step 3: POST the result to HTTPBin as a report

report_payload = {
    "report_title":   "Top 3 Most Populous Countries — South America",
    "generated_by":   "Session 4.1 Final Exercise",
    "countries":      top3
}

post_response = requests.___(
    "https://httpbin.org/post",
    ___=report_payload              # <-- params= or json=?
)

print("POST status code:", post_response.status_code)

# Step 4: Confirm the server received the data correctly
received = json.loads(post_response.json()["data"])

print()
print("Report received by server:")
print(f"  Title:   {received['report_title']}")
print(f"  Countries received: {len(received['countries'])}")
for c in received["countries"]:
    print(f"    - {c['name']} (pop: {c['population']:,})")

print()
print(f"Correct (status 200):          {post_response.status_code == 200}")
print(f"Correct (3 countries sent):    {len(received['countries']) == 3}")
print(f"Correct (Brazil in top 3):     {any(c['name'] == 'Brazil' for c in received['countries'])}")

---
## Session Summary

Here is everything we covered today:

### What is an API?
An API is a defined interface that lets one software system request data or actions from another. The restaurant waiter analogy: your code is the customer, the API is the waiter, the server is the kitchen.

### The Request-Response Cycle
Every API interaction: your program sends a **request** (URL + method + optional body), the server sends back a **response** (status code + optional body). Always. No exceptions.

### HTTP Methods

| Method | Purpose | Data location | Safe to repeat? |
|---|---|---|---|
| GET | Fetch data | URL parameters | Yes |
| POST | Create/submit data | Request body | No |

### URL Parameters
- **Path parameters** go inside the URL path: `/countries/india`
- **Query parameters** go after `?`: `?fields=name,population&page=1`
- Use `requests.get(url, params={...})` — let `requests` handle encoding

### REST Architecture — the four principles
1. **Everything is a resource** — URLs are nouns, methods are verbs
2. **Statelessness** — the server remembers nothing between requests; each request must be self-contained
3. **Uniform interface** — use HTTP methods for their intended purpose (GET to fetch, POST to create, DELETE to delete)
4. **Representations** — the server sends a representation of a resource (usually JSON), not the resource itself

---

### What comes next

**Session 4.2** will go deeper into the harder parts of real-world APIs:
- HTTP status codes and handling errors robustly
- Authentication and API keys
- Pagination — what to do when an API returns 10,000 results
- Request headers and how to set them

---

### Key Python used today

| Code | What it does |
|---|---|
| `requests.get(url)` | Makes a GET request |
| `requests.get(url, params={...})` | GET with query parameters |
| `requests.post(url, json={...})` | POST with a JSON body |
| `response.status_code` | The numeric status code (200, 404, etc.) |
| `response.json()` | Parse the response body as JSON into a Python object |
| `response.url` | The full URL that was actually called (after encoding) |
| `response.headers` | Dictionary of response headers |